merge_analyze_radiomics.ipynb: this file merges all the radiomics features into one file so that it can be easily accessed for modeling. Also does summary stats of radiomics features to make sure that everything looks okay. 

In [49]:
import pandas as pd
import os

In [63]:
keyphrase = '_results_2b_'
directory = '../radiomics'
labels = {'_results_1':'liver', '_results_2_':'spleen', '_results_2b_':'spleen3'}

df = pd.DataFrame()
for file in os.listdir():
    if keyphrase in file:
        print(file)
        try:
            data = pd.read_csv(file)
            df = pd.concat([df, data], axis=0)
        except:
            pass
            print(file)
to_keep = [col for col in df.columns if 'diagnostics' not in col]
df[to_keep].to_csv('radiomics_%s_1.csv' %labels[keyphrase], index=False)

radiomics_results_2b_5.csv
radiomics_results_2b_22.csv
radiomics_results_2b_19.csv
radiomics_results_2b_30.csv
radiomics_results_2b_38.csv
radiomics_results_2b_20.csv
radiomics_results_2b_35.csv
radiomics_results_2b_31.csv
radiomics_results_2b_58.csv
radiomics_results_2b_8.csv
radiomics_results_2b_40.csv
radiomics_results_2b_11.csv
radiomics_results_2b_43.csv
radiomics_results_2b_50.csv
radiomics_results_2b_21.csv
radiomics_results_2b_51.csv
radiomics_results_2b_18.csv
radiomics_results_2b_14.csv
radiomics_results_2b_33.csv
radiomics_results_2b_56.csv
radiomics_results_2b_41.csv
radiomics_results_2b_3.csv
radiomics_results_2b_36.csv
radiomics_results_2b_54.csv
radiomics_results_2b_23.csv
radiomics_results_2b_49.csv
radiomics_results_2b_24.csv
radiomics_results_2b_29.csv
radiomics_results_2b_34.csv
radiomics_results_2b_13.csv
radiomics_results_2b_46.csv
radiomics_results_2b_59.csv
radiomics_results_2b_15.csv
radiomics_results_2b_39.csv
radiomics_results_2b_17.csv
radiomics_results_2b_10

In [50]:
#df = pd.read_csv('radiomics_liver_1.csv')
spleen_df = pd.read_csv('radiomics_spleen.csv')
spleen_df = pd.read_csv('../predictCAD/cohorts/CADcohort_all_demo_rad.csv')

In [51]:
spleen_df.shape

(42543, 121)

In [52]:
df.info(verbose = True)

NameError: name 'df' is not defined

In [22]:
#print(len(df.ID.unique()))
#df_new = df.drop_duplicates('ID').dropna()
#print(len(df_new.ID.unique()))
print(len(spleen_df.ID.unique()))
#spleen_df_new = spleen_df.drop_duplicates('ID').dropna()
#print(len(spleen_df_new.ID.unique()))

42543


In [24]:
data = pd.read_csv("gs://ukbb_spleen/CHIPCAD_pheno_v5.csv", sep='\s+')
data = data.sort_values(by = ["ID"]).drop_duplicates()
data = data.drop_duplicates('ID')
print(data.shape)
#data_liver = pd.merge(data, df_new, on = "ID", how = "inner")
data_spleen = pd.merge(data, spleen_df, on = "ID", how = "inner")
#print(data_liver.shape)
print(data_spleen.shape)

(43543, 73)
(42543, 193)


In [ ]:
liver_IDs = set(data_liver.ID)
IDs = [pat for pat in data_spleen.ID if pat not in liver_IDs]

In [ ]:
len(IDs)
f = open('IDs_missing_liver.txt', 'w+')
for ID in IDs:
    f.write(str(ID)+"\n")
f.close()

In [ ]:
# there are 370 patients who did not have a spleen labelled to extract information from 
pd.set_option("display.max_rows", 10)
df_filt = df.dropna()
f = open('feature_stats/Radiomics_stats_%s.txt' %labels[keyphrase], 'w+')
for col in rad_feats:
    f.write(col+"\n")
    #print(df_filt[col].value_counts(dropna= False).sort_values('index'))
    f.write("Min %f \n" %df_filt[col].astype('float').min())
    f.write("Mean %f \n" %df_filt[col].astype('float').mean())
    f.write("Max %f \n \n" %df_filt[col].astype('float').max())

f.close()

In [53]:
# understand collinearity
label = "liver_demo"
spleen_pce_cols = ['original_firstorder_Entropy', 'original_firstorder_InterquartileRange',
        'original_shape_Maximum2DDiameterRow', 'original_glcm_Contrast', 
        'original_glcm_DifferenceAverage', 'original_glrlm_GrayLevelNonUniformity', 'original_shape_Sphericity']

liver_pce_cols = ['original_shape_MeshVolume', 'original_gldm_DependenceVariance', 'original_firstorder_Kurtosis', 
                  'original_gldm_GrayLevelNonUniformity', 'original_glszm_LargeAreaLowGrayLevelEmphasis', 
                 'original_shape_SurfaceVolumeRatio', 'original_shape_LeastAxisLength', 'original_glcm_MaximumProbability', 'original_shape_MajorAxisLength']

spleen_demo_cols = ['original_gldm_LargeDependenceHighGrayLevelEmphasis', 
                   'original_firstorder_Kurtosis', 'original_shape_Sphericity', 
                   'original_glszm_SmallAreaLowGrayLevelEmphasis', 
                  'original_glszm_GrayLevelNonUniformity', 'original_glcm_Id', 'original_shape_MeshVolume']

liver_demo_cols = ['original_glszm_GrayLevelNonUniformityNormalized', 'original_glcm_Idm', 'original_glcm_Idn', 
                  'original_gldm_DependenceVariance', 'original_glcm_SumSquares', 
                 'original_shape_LeastAxisLength', 'original_ngtdm_Busyness', 'original_glszm_LowGrayLevelZoneEmphasis',
                'original_shape_MajorAxisLength', 'original_glrlm_RunEntropy']

data = data_liver[liver_demo_cols].astype(float)
data.columns = data.columns.str.lstrip('original_') 
print(data.shape)
corr = data.corr().round(2)
corr.to_csv('feature_stats/feat_corr_map_%s.csv' %label)

NameError: name 'data_liver' is not defined

In [54]:
# correlate liver features with pre-existing liver features
data = pd.read_csv("gs://ukbb_spleen/ukb674757.tab", sep='\t+')
#data = data[['f.eid','f.21088.2.0', 'f.21089.2.0']]
data = data[['f.eid','f.21083.2.0', 'f.21170.2.0']]
data = data.rename(columns = {'f.eid':'ID','f.21083.2.0': 'spleen_vol', 'f.21170.2.0':'spleen_iron'})

#data = data.rename(columns = {'f.eid':'ID','f.21088.2.0': 'liver_fat', 'f.21089.2.0': 'liver_iron'})
data.spleen_vol.value_counts(dropna=False)

MemoryError: Unable to allocate 5.53 GiB for an array with shape (502359, 1477) and data type object

In [37]:
data_spleen_extra = pd.merge(data, data_spleen, on = "ID")

In [38]:
#data_spleen_extra = data_spleen_extra.dropna()
data_spleen_extra.shape

(42543, 194)

In [46]:
spleen_demo_cols = ['original_gldm_LargeDependenceHighGrayLevelEmphasis', 
                   'original_firstorder_Kurtosis', 'original_shape_Sphericity', 
                   'original_glszm_SmallAreaLowGrayLevelEmphasis', 
                  'original_glszm_GrayLevelNonUniformity', 'original_glcm_Id', 'original_shape_MeshVolume']

data = data_spleen_extra[['spleen_'+elem for elem in spleen_demo_cols]+['spleen_vol']].astype(float)
#data = data.add_prefix('spleen_') 
print(data.shape)
print(data.spleen_vol.value_counts())
corr = data.corr().round(2)
corr.to_csv('feature_stats/feat_corr_map_spleen_demo_extra_correct_coh.csv')

(42543, 8)
0.159691    12
0.125293    12
0.144939    11
0.151313    11
0.132030    11
            ..
0.303323     1
0.293464     1
0.097197     1
0.189501     1
0.247072     1
Name: spleen_vol, Length: 15354, dtype: int64


In [ ]:
corr['original_shape_MinorAxisLength']['original_shape_Maximum2DDiameterRow']

In [ ]:
import pickle5 as pickle
def load_pickle(pickle_path):
    with open(pickle_path, 'rb') as f:
        pkl_file = pickle.load(f)
    return pkl_file

orig_conversion_map = load_pickle("nnunet_data_conversion.pkl")
conversion_map = {}
for entry in orig_conversion_map:
    entry = entry[0]    
    for img_path in entry['img_paths']:
        conversion_map[img_path['orig']] = img_path['nnunet'] 
print(list(conversion_map.keys())[:10])
f = open('ID_paths_missing_liver.txt', 'w+')
counter = 0
for ID in IDs:
    for elem in ["opp", "inp", "fat", "wat"]:
        IDpath = "/home/jupyter/MRI-spleen/stitched_data/"+str(ID)+"/"+elem+".nii.gz"
        end = conversion_map[IDpath].split("/")[-1]
        frompath = "gs://ukbb_spleen/nnunet_data/"+end
        topath = conversion_map[IDpath]
        !gsutil -m cp "$frompath" "$topath"
    counter +=1
    if counter%100:
        print(counter)

In [ ]:
import pickle5 as pickle
def load_pickle(pickle_path):
    with open(pickle_path, 'rb') as f:
        pkl_file = pickle.load(f)
    return pkl_file

orig_conversion_map = load_pickle("nnunet_data_conversion.pkl")
conversion_map = {}
for entry in orig_conversion_map:
    entry = entry[0]    
    for img_path in entry['img_paths']:
        conversion_map[img_path['orig']] = img_path['nnunet'] 
print(list(conversion_map.keys())[:10])
f = open('ID_paths_missing_liver.txt', 'w+')
counter = 0
for ID in IDs:
    for elem in ["opp", "inp", "fat", "wat"]:
        IDpath = "/home/jupyter/MRI-spleen/stitched_data/"+str(ID)+"/"+elem+".nii.gz"
        end = conversion_map[IDpath].split("/")[-1]
        f.write(end+"\n")
        #frompath = "gs://ukbb_spleen/nnunet_data/"+end
        #topath = conversion_map[IDpath]
        #!gsutil -m cp "$frompath" "$topath"
    counter +=1
    if counter%100 == 0:
        print(counter)